# Corazón v2 — Día 2: semillas de la U-Net + ResNet-50 + Grad-CAM (E1, E2, E3, E4)

**Antes de correr (panel derecho de Kaggle):**
1. **Accelerator:** GPU T4 ×2 (este notebook usa las dos a la vez: U-Net en una, ResNet en la otra).
2. **Internet:** ON (clona el código de GitHub y baja los pesos de ImageNet de la ResNet-50).
3. **Add Input → Your Work → Notebooks** → el notebook del día 1, **Version 1**. Trae `data/` (los arrays ya
   preparados) y los pesos de la U-Net. **No hace falta** agregar NIH ni JSRT: todo está en `data/`.
4. **Save Version → Save & Run All (Commit)**. Se puede apagar la laptop.

**Qué corre** (estimado, no medido): GPU 0 → U-Net con semillas 1 y 2 (E1b y E1a) y el ajuste con contornos humanos
para las 3 semillas, ~2,5 h. GPU 1 → ResNet-50 × 3 semillas con PA + AP y × 3 solo con PA, más Grad-CAM, ~1–1,5 h.
La semilla 0 de la U-Net no se reentrena: se reevalúa con los pesos de la Version #1.

**Qué produce** (`/kaggle/working/`): `runs/` (pesos y predicciones) y `results/` con `E1_segmentacion.csv`,
`E2_diagnostico.csv`, `E3_acuerdo_ict.csv`, `E4_por_vista.csv`, `E4_gradcam.csv` y figuras de Grad-CAM.

In [ ]:
import glob, json, os, shutil, subprocess, sys, threading, time
T0 = time.time()
W = "/kaggle/working"
REPO = f"{W}/repo"
REPO_URL = "https://github.com/loupv99/corazon-v2"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
os.chdir(REPO)
print("código:", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"])
import torch
N_GPU = torch.cuda.device_count()
print("GPUs:", N_GPU, "| CPUs:", os.cpu_count())

In [ ]:
# Ubicar el Output de la Version #1 (sin recorrer /kaggle/input entero: con NIH montado tardaría minutos)
def buscar(rel):
    for prof in range(5):
        r = sorted(glob.glob("/kaggle/input/" + "*/" * prof + rel))
        if r:
            return r[0]
    return ""

npy = buscar("data/nih_images.npy")
assert npy, "No encuentro data/nih_images.npy: agrega como input el Output del notebook del día 1 (Version 1)"
V1 = os.path.dirname(os.path.dirname(npy))
DATA = f"{V1}/data"
print("Version #1:", V1)
print("data:", sorted(os.listdir(DATA)))

# Semilla 0 de la U-Net = pesos de la Version #1 (se reevalúan; así se obtiene también el ICT por placa de SCR, E3)
RUNS = f"{W}/runs"
for var in ("E1b_debil_rca07", "E1a_debil_todas"):
    origen = glob.glob(f"{V1}/runs/{var}/best*")
    assert origen, f"falta {V1}/runs/{var}/best.pt"
    os.makedirs(f"{RUNS}/{var}_s0", exist_ok=True)
    shutil.copy(origen[0], f"{RUNS}/{var}_s0/best.pt")
    if os.path.exists(f"{V1}/runs/{var}/results.json"):
        shutil.copy(f"{V1}/runs/{var}/results.json", f"{RUNS}/{var}_s0/results_entrenamiento_v1.json")
print(sorted(os.listdir(RUNS)))

## Dos colas en paralelo
- **GPU 0, U-Net (E1 y E3).** Reevalúa la semilla 0. Ajusta con contornos humanos (E1c) partiendo de cada semilla de
  E1b; ahora el mejor epoch se elige con 20 placas doradas. Entrena E1b y E1a con las semillas 1 y 2.
- **GPU 1, ResNet-50 (E2 y E4).** 3 semillas con PA + AP y 3 solo con PA, más Grad-CAM de cada una sobre el test
  oficial.

Si una tarea falla, la cola sigue con las demás y al final se listan las fallas.

In [ ]:
SEG = ["src.train_seg", "--data", DATA, "--out", RUNS]
CLS = ["src.classifier", "--data", DATA, "--out", RUNS]
def ft(s):   # E1c: ajuste con contornos humanos desde E1b de la misma semilla
    return SEG + ["--run", f"E1c_rca07_ft_scr_s{s}", "--train-source", "scr", "--init",
                  f"{RUNS}/E1b_debil_rca07_s{s}/best.pt", "--epochs", "30", "--lr", "2e-4", "--bs", "8", "--seed", str(s)]

cola_unet = [SEG + ["--run", "E1b_debil_rca07_s0", "--epochs", "0"],
             SEG + ["--run", "E1a_debil_todas_s0", "--epochs", "0"],
             ft(0)]
for s in (1, 2):
    cola_unet += [SEG + ["--run", f"E1b_debil_rca07_s{s}", "--masks", "rca07", "--epochs", "30", "--seed", str(s)], ft(s)]
for s in (1, 2):
    cola_unet += [SEG + ["--run", f"E1a_debil_todas_s{s}", "--masks", "all", "--epochs", "30", "--seed", str(s)]]

cola_resnet = []
for s in (0, 1, 2):
    cola_resnet += [CLS + ["--run", f"E2_resnet_paap_s{s}", "--seed", str(s)],
                    ["src.gradcam", "--data", DATA, "--ckpt", f"{RUNS}/E2_resnet_paap_s{s}/best.pt",
                     "--out", f"{RUNS}/E2_resnet_paap_s{s}"]]
for s in (0, 1, 2):
    cola_resnet += [CLS + ["--run", f"E4_resnet_pa_s{s}", "--views", "PA", "--seed", str(s)],
                    ["src.gradcam", "--data", DATA, "--ckpt", f"{RUNS}/E4_resnet_pa_s{s}/best.pt",
                     "--out", f"{RUNS}/E4_resnet_pa_s{s}"]]

fallas, tiempos = [], []
def correr(nombre, gpu, trabajos):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1")
    for args in trabajos:
        t = time.time()
        etiqueta = args[args.index("--run") + 1] if "--run" in args else f"gradcam {os.path.basename(args[-1])}"
        p = subprocess.Popen([sys.executable, "-m", *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, env=env)
        for linea in p.stdout:
            print(f"[{nombre}] {linea}", end="", flush=True)
        ok = p.wait() == 0
        if not ok:
            fallas.append(etiqueta)
        tiempos.append({"cola": nombre, "tarea": etiqueta, "ok": ok, "min": round((time.time() - t) / 60, 1)})
        print(f"[{nombre}] --- {etiqueta} {'OK' if ok else 'FALLÓ'} en {(time.time() - t) / 60:.1f} min "
              f"(total {(time.time() - T0) / 60:.0f} min)", flush=True)

if N_GPU >= 2:
    hilos = [threading.Thread(target=correr, args=("unet", 0, cola_unet)),
             threading.Thread(target=correr, args=("resnet", 1, cola_resnet))]
    for h in hilos: h.start()
    for h in hilos: h.join()
else:   # una sola GPU: una cola detrás de la otra
    correr("resnet", 0, cola_resnet)
    correr("unet", 0, cola_unet)
print("fallas:", fallas or "ninguna", f"| {(time.time() - T0) / 60:.0f} min")

## Tablas para el paper

In [ ]:
import pandas as pd
from IPython.display import Image, display
os.makedirs(f"{W}/results", exist_ok=True)
subprocess.run([sys.executable, "-W", "ignore", "-m", "src.analisis", "--data", DATA, "--runs", RUNS,
                "--out", f"{W}/results"], check=True)
pd.DataFrame(tiempos).to_csv(f"{W}/results/tiempos_gpu.csv", index=False)   # R6: horas de GPU
for f in sorted(glob.glob(f"{RUNS}/*/gradcam_ejemplos.png"))[:2]:
    print(f); display(Image(f))
    shutil.copy(f, f"{W}/results/{os.path.basename(os.path.dirname(f))}_gradcam.png")

In [ ]:
for d in glob.glob(f"{RUNS}/*/last.pt"): os.remove(d)   # los best.pt bastan
print("results/:", sorted(os.listdir(f"{W}/results")))
print("fallas:", fallas or "ninguna", f"| tiempo total {(time.time() - T0) / 60:.0f} min")